# Checkpoint — Assistente Texto → SQL com Qwen3-0.6B

**Grupo:** Nome 1 (RM) · Nome 2 (RM) · Nome 3 (RM) · Nome 4 (RM) · Nome 5 (RM)

**Problema:** quem não sabe SQL não consegue tirar informações de um banco de dados. Este projeto treina uma LLM pequena para **transformar perguntas em português em consultas SQL** para o banco fixo de uma loja online. A consulta gerada é **executada** e o resultado volta para o usuário.

```
Pergunta:  "Quanto faturamos em março de 2025?"
SQL:       SELECT SUM(i.quantidade * i.preco_unitario) FROM itens_pedido i JOIN pedidos pe ...
Resultado: 1.284.530,17
```

| Item da entrega | Onde está neste notebook |
|---|---|
| 1. Escolher a LLM | Etapa 3 e Etapa 6: **Qwen3-0.6B** (da lista do professor) |
| 2. Problema específico | Etapas 4 e 5: Texto → SQL em um schema fixo (sugestão da lista) |
| 3. Testar parâmetros | Etapa 10: temperature e top_p na geração, taxa de aprendizado e épocas no treino |
| 4. Avaliar o modelo | Etapas 7 e 9: o SQL é **executado** e o resultado comparado com o gabarito, antes e depois do treino |
| 5. Testes na GPU | Etapa 1 (verificação) e todo o treino/inferência em CUDA |
| 6. Frontend | Etapa 13: interface web (Gradio) para o usuário fazer perguntas |
| 7. Print da GPU | Saída da Etapa 1 |
| 8. GitHub | Este notebook (com as saídas) + README no repositório |

## Como rodar
1. Abra este notebook no **Google Colab**.
2. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** → Salvar.
3. Rode as células **de cima para baixo** (Shift+Enter). O treino (Etapa 8) leva cerca de 10 a 20 minutos.

## ETAPA 1 — Verificar a GPU (itens 5 e 7: tire o print desta saída)

In [ ]:
!nvidia-smi

import platform
import torch

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB")
    print("✅ GPU funcionando com CUDA")
else:
    print("⚠️ Sem GPU! Vá em Ambiente de execução → Alterar o tipo → GPU T4")

## ETAPA 2 — Instalar as dependências

In [ ]:
# O Colab já vem com PyTorch + CUDA. O Qwen3 precisa do transformers 4.51 ou mais novo.
!pip install -q -U "transformers>=4.51" accelerate gradio

## ETAPA 3 — Funções compartilhadas
Mesmo formato da aula: um **prompt padronizado** (`### Banco / ### Pergunta / ### SQL`), uma função para **carregar** o modelo e outra para **gerar** a resposta. O esquema do banco vai dentro do prompt, para o modelo saber quais tabelas e colunas existem.

In [ ]:
"""Funções compartilhadas pelas etapas do notebook."""

import gc
import json
import random
import re
import time
from pathlib import Path

import torch

PASTA = Path.cwd()
BASE = "Qwen/Qwen3-0.6B"                    # LLM escolhida (item 1)
TREINADO = PASTA / "modelo-sql-treinado"    # onde o modelo treinado é salvo
DADOS = PASTA / "dados"
RESULTADOS = PASTA / "resultados"
DADOS.mkdir(exist_ok=True)
RESULTADOS.mkdir(exist_ok=True)

ESQUEMA = (
    "clientes(id, nome, cidade, estado, data_cadastro)\n"
    "categorias(id, nome)\n"
    "produtos(id, nome, categoria_id, preco, estoque)\n"
    "pedidos(id, cliente_id, data_pedido, status)\n"
    "itens_pedido(id, pedido_id, produto_id, quantidade, preco_unitario)"
)


def prompt(pergunta):
    """Monta o mesmo formato de prompt usado no treinamento."""
    return (
        f"### Banco:\n{ESQUEMA}\n"
        f"### Pergunta:\n{pergunta}\n"
        "### SQL:\n"
    )


def carregar(caminho):
    """Carrega o modelo base ou o modelo treinado."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tok = AutoTokenizer.from_pretrained(str(caminho))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"  # necessário para gerar várias respostas de uma vez

    model = AutoModelForCausalLM.from_pretrained(
        str(caminho), torch_dtype=torch.float32
    ).to(device)
    model.eval()

    print("Modelo:", caminho)
    print("Hardware:", device)
    return model, tok, device


@torch.no_grad()
def gerar(model, tok, device, textos, temperatura=0.0, top_p=1.0, max_tokens=120):
    """Gera a resposta da LLM para um prompt (ou uma lista de prompts)."""
    unico = isinstance(textos, str)
    if unico:
        textos = [textos]

    entrada = tok(
        textos, return_tensors="pt", padding=True, add_special_tokens=False
    ).to(device)

    params = dict(
        max_new_tokens=max_tokens,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if temperatura > 0:
        params.update(do_sample=True, temperature=temperatura, top_p=top_p, top_k=0)
    else:
        params.update(do_sample=False)  # sempre o token mais provável

    saida = model.generate(**entrada, **params)
    novos = saida[:, entrada["input_ids"].shape[1]:]
    respostas = tok.batch_decode(novos, skip_special_tokens=True)
    return respostas[0] if unico else respostas


def extrair_sql(texto):
    """Pega só a primeira consulta SQL do texto gerado."""
    bloco = re.search(r"```(?:sql)?\s*(.*?)(?:```|$)", texto, re.S | re.I)
    sql = (bloco.group(1) if bloco else texto).strip()
    inicio = re.search(r"\b(SELECT|WITH)\b", sql, re.I)
    if inicio:
        sql = sql[inicio.start():]
    sql = sql.split(";")[0].strip()
    return sql + ";" if sql else ""


print("Funções carregadas. Modelo base:", BASE)

## ETAPA 4 — Criar o banco de dados da loja (o "schema fixo")
Um banco SQLite com 5 tabelas, gerado sempre igual (semente fixa). As consultas rodam em **modo somente leitura**, então o modelo não consegue apagar nem alterar dados.

**Regra de negócio** que o modelo precisa aprender: *faturamento e vendas não contam pedidos com status `cancelado`*. Essa regra **não está no prompt**; o modelo só aprende com os exemplos do treino.

In [ ]:
import sqlite3
from datetime import date, timedelta

BANCO = DADOS / "loja.db"

CATEGORIAS = {
    "Eletrônicos": (["Smart TV 50 polegadas", "Fone Bluetooth", "Caixa de Som Portátil", "Soundbar",
                     "Projetor", "Câmera Digital", "Videogame"], (150, 4000)),
    "Informática": (["Notebook", "Mouse Sem Fio", "Teclado Mecânico", "Monitor 24 polegadas",
                     "SSD 1TB", "Impressora", "Webcam"], (60, 5000)),
    "Celulares": (["Smartphone Básico", "Smartphone Intermediário", "Smartphone Premium",
                   "Carregador Turbo", "Capinha", "Película de Vidro", "Smartwatch"], (25, 6000)),
    "Eletrodomésticos": (["Geladeira", "Micro-ondas", "Air Fryer", "Liquidificador", "Cafeteira",
                          "Aspirador de Pó", "Ventilador"], (90, 4500)),
    "Móveis": (["Sofá 3 Lugares", "Mesa de Jantar", "Cadeira Gamer", "Guarda-Roupa", "Estante",
                "Escrivaninha", "Cama Box"], (200, 3500)),
    "Esporte": (["Bicicleta", "Esteira", "Halteres 10kg", "Tênis de Corrida", "Bola de Futebol",
                 "Tapete de Yoga", "Garrafa Térmica"], (30, 3000)),
    "Livros": (["Romance Best-seller", "Livro de Receitas", "Guia de Python", "Box Fantasia",
                "Biografia", "Livro Infantil", "Dicionário"], (20, 250)),
    "Brinquedos": (["Quebra-Cabeça 1000 peças", "Boneca", "Carrinho de Controle", "Jogo de Tabuleiro",
                    "Blocos de Montar", "Pelúcia", "Patinete"], (30, 600)),
}
CIDADES = {
    "SP": ["São Paulo", "Campinas", "Santos"], "RJ": ["Rio de Janeiro", "Niterói"],
    "MG": ["Belo Horizonte", "Uberlândia"], "PR": ["Curitiba", "Londrina"],
    "RS": ["Porto Alegre", "Caxias do Sul"], "BA": ["Salvador", "Feira de Santana"],
    "PE": ["Recife", "Olinda"], "SC": ["Florianópolis", "Joinville"],
    "GO": ["Goiânia"], "CE": ["Fortaleza"],
}
NOMES = ["Ana", "Bruno", "Carla", "Diego", "Eduarda", "Felipe", "Gabriela", "Henrique", "Isabela", "João",
         "Larissa", "Lucas", "Mariana", "Matheus", "Natália", "Otávio", "Patrícia", "Rafael", "Sofia", "Thiago",
         "Vanessa", "Vinícius", "Beatriz", "Gustavo", "Juliana", "Pedro", "Camila", "Rodrigo", "Letícia", "André"]
SOBRENOMES = ["Silva", "Santos", "Oliveira", "Souza", "Lima", "Pereira", "Costa", "Ferreira", "Almeida",
              "Ribeiro", "Carvalho", "Gomes", "Martins", "Rocha", "Barbosa", "Araújo", "Mendes", "Nunes"]
STATUS = ["entregue", "enviado", "pendente", "cancelado"]


def criar_banco(semente=42):
    if BANCO.exists():
        BANCO.unlink()
    rng = random.Random(semente)
    data_aleatoria = lambda ini, fim: (ini + timedelta(days=rng.randint(0, (fim - ini).days))).isoformat()
    con = sqlite3.connect(BANCO)
    con.executescript("""
        CREATE TABLE clientes (id INTEGER PRIMARY KEY, nome TEXT, cidade TEXT, estado TEXT, data_cadastro DATE);
        CREATE TABLE categorias (id INTEGER PRIMARY KEY, nome TEXT);
        CREATE TABLE produtos (id INTEGER PRIMARY KEY, nome TEXT, categoria_id INTEGER, preco REAL, estoque INTEGER);
        CREATE TABLE pedidos (id INTEGER PRIMARY KEY, cliente_id INTEGER, data_pedido DATE, status TEXT);
        CREATE TABLE itens_pedido (id INTEGER PRIMARY KEY, pedido_id INTEGER, produto_id INTEGER,
                                   quantidade INTEGER, preco_unitario REAL);
    """)
    produtos, pid = [], 0
    for cid, (cat, (nomes, (pmin, pmax))) in enumerate(CATEGORIAS.items(), start=1):
        con.execute("INSERT INTO categorias VALUES (?, ?)", (cid, cat))
        for j, nome in enumerate(nomes):
            pid += 1
            preco = round(rng.uniform(pmin, pmax), 2)
            con.execute("INSERT INTO produtos VALUES (?, ?, ?, ?, ?)", (pid, nome, cid, preco, rng.randint(0, 120)))
            produtos.append((pid, preco, j != len(nomes) - 1))  # o último de cada categoria nunca vende
    usados = set()
    for cid in range(1, 301):
        while True:
            nome = f"{rng.choice(NOMES)} {rng.choice(SOBRENOMES)} {rng.choice(SOBRENOMES)}"
            if nome not in usados:
                usados.add(nome)
                break
        uf = rng.choice(list(CIDADES))
        con.execute("INSERT INTO clientes VALUES (?, ?, ?, ?, ?)",
                    (cid, nome, rng.choice(CIDADES[uf]), uf, data_aleatoria(date(2022, 1, 1), date(2025, 6, 30))))
    vendaveis = [p for p in produtos if p[2]]
    item = 0
    for ped in range(1, 2001):
        con.execute("INSERT INTO pedidos VALUES (?, ?, ?, ?)",
                    (ped, rng.randint(1, 270), data_aleatoria(date(2025, 1, 1), date(2025, 12, 31)),
                     rng.choices(STATUS, [0.6, 0.15, 0.1, 0.15])[0]))  # clientes 271-300 nunca compram
        for prod_id, preco, _ in rng.sample(vendaveis, rng.randint(1, 4)):
            item += 1
            con.execute("INSERT INTO itens_pedido VALUES (?, ?, ?, ?, ?)",
                        (item, ped, prod_id, rng.randint(1, 3), round(preco * rng.uniform(0.9, 1.0), 2)))
    con.commit()
    con.close()


def executar(sql, max_linhas=1000, timeout_s=5.0):
    """Executa a consulta em modo SOMENTE LEITURA."""
    con = sqlite3.connect(f"file:{BANCO.resolve().as_posix()}?mode=ro", uri=True)
    limite = time.monotonic() + timeout_s
    con.set_progress_handler(lambda: 1 if time.monotonic() > limite else 0, 10_000)
    try:
        cur = con.execute(sql)
        colunas = [d[0] for d in cur.description] if cur.description else []
        return colunas, cur.fetchmany(max_linhas)
    finally:
        con.close()


def mesmo_resultado(sql_gerado, sql_correto):
    """Retorna (sql_valido, resultado_correto). Compara o RESULTADO, não o texto do SQL."""
    try:
        _, gerado = executar(sql_gerado)
    except Exception:
        return False, False
    _, correto = executar(sql_correto)
    norm = lambda linhas: [tuple(round(v, 2) if isinstance(v, float) else v for v in l) for l in linhas]
    gerado, correto = norm(gerado), norm(correto)
    if "order by" in sql_correto.lower():
        return True, gerado == correto
    return True, sorted(map(repr, gerado)) == sorted(map(repr, correto))


criar_banco()
for tabela in ["clientes", "categorias", "produtos", "pedidos", "itens_pedido"]:
    print(f"{tabela:13s}: {executar(f'SELECT COUNT(*) FROM {tabela}')[1][0][0]} linhas")
print("\nExemplo:", executar("SELECT nome, preco FROM produtos ORDER BY preco DESC LIMIT 3")[1])

## ETAPA 5 — Criar os dados de treinamento, validação e teste
Assim como na aula, cada exemplo é um par **(pergunta, SQL correto)**. São 26 tipos de pergunta (templates), cada um com 4 formas de perguntar, preenchidas com valores reais do banco.

* **Treino:** usa 3 das 4 formas de perguntar.
* **Validação e teste:** usam a 4ª forma, que o modelo **nunca viu** no treino, com outros valores.
* **Teste manual:** 30 perguntas escritas à mão, de forma bem diferente, para medir a robustez.

In [ ]:
MESES = {"janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04", "maio": "05", "junho": "06",
         "julho": "07", "agosto": "08", "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12"}
CLIENTES = [r[0] for r in executar("SELECT nome FROM clientes")[1]]

FAT = "SUM(i.quantidade * i.preco_unitario)"
J_CAT = "FROM produtos p JOIN categorias c ON p.categoria_id = c.id"
J_VENDAS = "FROM itens_pedido i JOIN pedidos pe ON i.pedido_id = pe.id"
J_CLI = "FROM clientes cl JOIN pedidos pe ON pe.cliente_id = cl.id JOIN itens_pedido i ON i.pedido_id = pe.id"
OK = "pe.status <> 'cancelado'"

# Sorteio dos valores que preenchem as perguntas.
P = {
    "cat": lambda r: {"cat": r.choice(list(CATEGORIAS))},
    "uf": lambda r: {"uf": r.choice(list(CIDADES))},
    "cidade": lambda r: {"cidade": r.choice([c for l in CIDADES.values() for c in l])},
    "status": lambda r: {"status": r.choice(STATUS)},
    "mes": lambda r: (lambda m: {"mes": m, "mm": MESES[m]})(r.choice(list(MESES))),
    "k": lambda r: {"k": r.choice([3, 5, 10])},
    "n_preco": lambda r: {"n": r.choice([50, 100, 200, 500, 1000, 2000, 3000])},
    "faixa": lambda r: (lambda a: {"a": a, "b": a * r.choice([2, 3, 5])})(r.choice([50, 100, 200, 500, 1000])),
    "n_estoque": lambda r: {"n": r.choice([5, 10, 20, 30])},
    "cliente": lambda r: {"cliente": r.choice(CLIENTES)},
    "ano": lambda r: {"ano": r.choice([2022, 2023, 2024, 2025])},
}

# Gabarito: o SQL correto de cada tipo de pergunta. A ÚLTIMA forma de perguntar fica fora do treino.
TEMPLATES = [
    ("produtos_categoria", "cat", ["Quais produtos são da categoria {cat}?", "Liste os produtos de {cat}.",
        "Mostre os nomes dos produtos da categoria {cat}.", "Quais itens temos em {cat}?"],
     f"SELECT p.nome {J_CAT} WHERE c.nome = '{{cat}}';"),
    ("qtd_produtos_categoria", "cat", ["Quantos produtos existem na categoria {cat}?",
        "Qual a quantidade de produtos de {cat}?", "Conte os produtos da categoria {cat}.",
        "Quantos itens diferentes temos em {cat}?"],
     f"SELECT COUNT(*) {J_CAT} WHERE c.nome = '{{cat}}';"),
    ("preco_acima", "n_preco", ["Quais produtos custam mais de {n} reais?", "Liste os produtos com preço acima de R$ {n}.",
        "Mostre os produtos com preço maior que {n}.", "Que produtos passam de {n} reais?"],
     "SELECT nome, preco FROM produtos WHERE preco > {n};"),
    ("preco_entre", "faixa", ["Quais produtos custam entre {a} e {b} reais, ordenados pelo preço?",
        "Liste os produtos com preço de R$ {a} a R$ {b}, do mais barato ao mais caro.",
        "Mostre os produtos na faixa de {a} a {b} reais em ordem crescente de preço.",
        "Produtos entre {a} e {b} reais, do menor para o maior preço."],
     "SELECT nome, preco FROM produtos WHERE preco BETWEEN {a} AND {b} ORDER BY preco;"),
    ("estoque_baixo", "n_estoque", ["Quais produtos têm menos de {n} unidades em estoque?",
        "Liste os produtos com estoque abaixo de {n}.", "Que produtos estão com estoque menor que {n}?",
        "Mostre os produtos com menos de {n} no estoque."],
     "SELECT nome, estoque FROM produtos WHERE estoque < {n};"),
    ("top_caros", "k", ["Quais são os {k} produtos mais caros?", "Liste os {k} produtos de maior preço.",
        "Me mostre o top {k} produtos mais caros.", "Quais os {k} produtos com o preço mais alto?"],
     "SELECT nome, preco FROM produtos ORDER BY preco DESC LIMIT {k};"),
    ("clientes_estado", "uf", ["Quais clientes são do estado {uf}?", "Liste os clientes de {uf}.",
        "Mostre os nomes dos clientes que moram em {uf}.", "Quem são os clientes do {uf}?"],
     "SELECT nome FROM clientes WHERE estado = '{uf}';"),
    ("clientes_por_estado", None, ["Quantos clientes temos em cada estado?", "Qual o número de clientes por estado?",
        "Conte os clientes agrupados por estado.", "Mostre a quantidade de clientes de cada UF."],
     "SELECT estado, COUNT(*) AS total FROM clientes GROUP BY estado;"),
    ("clientes_cidade", "cidade", ["Quantos clientes temos em {cidade}?", "Qual o número de clientes da cidade de {cidade}?",
        "Conte os clientes que moram em {cidade}.", "Quantas pessoas de {cidade} são clientes?"],
     "SELECT COUNT(*) FROM clientes WHERE cidade = '{cidade}';"),
    ("pedidos_status", "status", ["Quantos pedidos estão com status {status}?", "Conte os pedidos com status {status}.",
        "Qual o total de pedidos no status {status}?", "Quantos pedidos têm o status '{status}'?"],
     "SELECT COUNT(*) FROM pedidos WHERE status = '{status}';"),
    ("pedidos_por_status", None, ["Quantos pedidos existem em cada status?", "Mostre a quantidade de pedidos por status.",
        "Conte os pedidos agrupando por status.", "Qual a distribuição dos pedidos por status?"],
     "SELECT status, COUNT(*) AS total FROM pedidos GROUP BY status;"),
    ("pedidos_mes", "mes", ["Quantos pedidos foram feitos em {mes} de 2025?", "Qual o número de pedidos de {mes}/2025?",
        "Conte os pedidos realizados no mês de {mes} de 2025.", "Quantos pedidos tivemos em {mes} de 2025?"],
     "SELECT COUNT(*) FROM pedidos WHERE strftime('%Y-%m', data_pedido) = '2025-{mm}';"),
    ("faturamento_mes", "mes", ["Qual foi o faturamento de {mes} de 2025?", "Quanto vendemos em {mes} de 2025?",
        "Qual a receita total do mês de {mes}/2025?", "Quanto faturamos em {mes} de 2025?"],
     f"SELECT {FAT} {J_VENDAS} WHERE {OK} AND strftime('%Y-%m', pe.data_pedido) = '2025-{{mm}}';"),
    ("faturamento_categoria", None, ["Qual o faturamento por categoria?", "Quanto cada categoria faturou?",
        "Mostre a receita total de cada categoria.", "Qual a receita de vendas por categoria?"],
     f"SELECT c.nome, {FAT} AS faturamento {J_VENDAS} JOIN produtos p ON i.produto_id = p.id "
     f"JOIN categorias c ON p.categoria_id = c.id WHERE {OK} GROUP BY c.nome;"),
    ("top_vendidos", "k", ["Quais são os {k} produtos mais vendidos?", "Liste os {k} produtos com mais unidades vendidas.",
        "Top {k} produtos em quantidade vendida.", "Quais os {k} produtos que mais venderam?"],
     f"SELECT p.nome, SUM(i.quantidade) AS unidades {J_VENDAS} JOIN produtos p ON i.produto_id = p.id "
     f"WHERE {OK} GROUP BY p.id ORDER BY unidades DESC LIMIT {{k}};"),
    ("top_clientes", "k", ["Quais os {k} clientes que mais gastaram?", "Liste os {k} clientes com maior valor em compras.",
        "Top {k} clientes por valor comprado.", "Quem são os {k} clientes que mais compraram em reais?"],
     f"SELECT cl.nome, {FAT} AS total {J_CLI} WHERE {OK} GROUP BY cl.id ORDER BY total DESC LIMIT {{k}};"),
    ("ticket_medio", None, ["Qual o ticket médio dos pedidos?", "Qual o valor médio de um pedido?",
        "Quanto custa em média cada pedido?", "Qual a média de valor por pedido?"],
     f"SELECT AVG(total) FROM (SELECT pe.id, {FAT} AS total {J_VENDAS} WHERE {OK} GROUP BY pe.id);"),
    ("clientes_sem_pedido", None, ["Quais clientes nunca fizeram um pedido?", "Liste os clientes sem nenhuma compra.",
        "Mostre os clientes que ainda não compraram nada.", "Quem são os clientes que nunca pediram nada?"],
     "SELECT nome FROM clientes WHERE id NOT IN (SELECT cliente_id FROM pedidos);"),
    ("produtos_nunca_vendidos", None, ["Quais produtos nunca foram vendidos?", "Liste os produtos sem nenhuma venda.",
        "Que produtos não aparecem em nenhum pedido?", "Quais produtos nunca saíram?"],
     "SELECT nome FROM produtos WHERE id NOT IN (SELECT produto_id FROM itens_pedido);"),
    ("pedidos_cliente", "cliente", ["Quantos pedidos o cliente {cliente} fez?", "Qual o número de pedidos de {cliente}?",
        "Conte os pedidos feitos por {cliente}.", "Quantas compras {cliente} já fez?"],
     "SELECT COUNT(*) FROM pedidos pe JOIN clientes cl ON pe.cliente_id = cl.id WHERE cl.nome = '{cliente}';"),
    ("pedidos_por_mes", None, ["Quantos pedidos foram feitos em cada mês de 2025?",
        "Mostre o número de pedidos por mês em 2025.", "Conte os pedidos de 2025 agrupados por mês.",
        "Qual a quantidade mensal de pedidos em 2025?"],
     "SELECT strftime('%m', data_pedido) AS mes, COUNT(*) AS total FROM pedidos "
     "WHERE strftime('%Y', data_pedido) = '2025' GROUP BY mes;"),
    ("preco_medio_por_categoria", None, ["Qual o preço médio dos produtos de cada categoria?",
        "Mostre a média de preço por categoria.", "Calcule o preço médio agrupado por categoria.",
        "Qual a média de preços em cada categoria?"],
     f"SELECT c.nome, AVG(p.preco) AS preco_medio {J_CAT} GROUP BY c.nome;"),
    ("preco_medio_categoria", "cat", ["Qual o preço médio dos produtos de {cat}?", "Qual a média de preço da categoria {cat}?",
        "Calcule o preço médio em {cat}.", "Quanto custa em média um produto de {cat}?"],
     f"SELECT AVG(p.preco) {J_CAT} WHERE c.nome = '{{cat}}';"),
    ("clientes_cadastrados_ano", "ano", ["Quantos clientes se cadastraram em {ano}?",
        "Qual o número de novos clientes em {ano}?", "Conte os clientes cadastrados no ano de {ano}.",
        "Quantos cadastros de clientes tivemos em {ano}?"],
     "SELECT COUNT(*) FROM clientes WHERE strftime('%Y', data_cadastro) = '{ano}';"),
    ("estado_maior_faturamento", None, ["Qual estado gerou o maior faturamento?", "Em qual UF vendemos mais?",
        "Qual o estado com mais receita?", "Qual estado mais faturou?"],
     f"SELECT cl.estado, {FAT} AS faturamento {J_CLI} WHERE {OK} GROUP BY cl.estado ORDER BY faturamento DESC LIMIT 1;"),
    ("faturamento_estado", "uf", ["Qual o faturamento com clientes de {uf}?",
        "Quanto os clientes do estado {uf} compraram no total?", "Qual a receita vinda de {uf}?",
        "Quanto vendemos para clientes de {uf}?"],
     f"SELECT {FAT} {J_CLI} WHERE {OK} AND cl.estado = '{{uf}}';"),
]
SQL_DO = {t[0]: t[3] for t in TEMPLATES}

rng = random.Random(42)
grupos = {"treino": [], "validacao": [], "teste": []}

for tid, gerador, frases, sql in TEMPLATES:
    def novo(frase):
        params = gerador and P[gerador](rng) or {}
        texto = frase.format(**params)
        if rng.random() < 0.3:             # pequenas variações de escrita
            texto = texto[0].lower() + texto[1:]
        return {"template": tid, "pergunta": texto, "sql": sql.format(**params)}

    for _ in range(60 if gerador else 15):
        grupos["treino"].append(novo(rng.choice(frases[:-1])))
    vistos = set()
    for grupo, qtd in [("validacao", 3), ("teste", 6)]:
        for _ in range(qtd if gerador else (0 if grupo == "validacao" else 1)):
            ex = novo(frases[-1])
            if ex["pergunta"] not in vistos:
                vistos.add(ex["pergunta"])
                grupos[grupo].append(ex)

rng.shuffle(grupos["treino"])

# Teste manual: perguntas escritas à mão, bem diferentes das do treino.
MANUAL = [
    ("Me diz quantos produtos de Livros a gente tem", "qtd_produtos_categoria", {"cat": "Livros"}),
    ("quais produtos de informática?", "produtos_categoria", {"cat": "Informática"}),
    ("Preciso da lista de produtos que custam acima de 1000 reais, com o preço", "preco_acima", {"n": 1000}),
    ("Quais itens estão acabando? Quero os que têm menos de 10 unidades", "estoque_baixo", {"n": 10}),
    ("Quero ver os 5 produtos com o preço mais alto", "top_caros", {"k": 5}),
    ("Quem são os clientes do RJ?", "clientes_estado", {"uf": "RJ"}),
    ("Lista de clientes de Pernambuco", "clientes_estado", {"uf": "PE"}),
    ("Como os clientes estão distribuídos por estado?", "clientes_por_estado", {}),
    ("Quantas pessoas cadastradas moram em Curitiba?", "clientes_cidade", {"cidade": "Curitiba"}),
    ("Quantos pedidos foram cancelados?", "pedidos_status", {"status": "cancelado"}),
    ("Quantos pedidos ainda estão pendentes?", "pedidos_status", {"status": "pendente"}),
    ("Quantos pedidos foram entregues?", "pedidos_status", {"status": "entregue"}),
    ("Quantos pedidos tivemos em março?", "pedidos_mes", {"mm": "03"}),
    ("Quanto faturamos em dezembro de 2025?", "faturamento_mes", {"mm": "12"}),
    ("Total vendido em julho/2025", "faturamento_mes", {"mm": "07"}),
    ("Receita por categoria", "faturamento_categoria", {}),
    ("Top 3 produtos que mais venderam", "top_vendidos", {"k": 3}),
    ("Quais os 10 melhores clientes em valor gasto?", "top_clientes", {"k": 10}),
    ("Valor médio por pedido", "ticket_medio", {}),
    ("Tem algum cliente cadastrado que nunca comprou?", "clientes_sem_pedido", {}),
    ("Existe algum produto encalhado, que nunca vendeu?", "produtos_nunca_vendidos", {}),
    ("Evolução mensal da quantidade de pedidos em 2025", "pedidos_por_mes", {}),
    ("Média de preço de cada categoria", "preco_medio_por_categoria", {}),
    ("Qual o preço médio dos Celulares?", "preco_medio_categoria", {"cat": "Celulares"}),
    ("Quantos clientes novos entraram em 2024?", "clientes_cadastrados_ano", {"ano": 2024}),
    ("Qual UF trouxe mais dinheiro?", "estado_maior_faturamento", {}),
    ("Quanto os clientes de MG gastaram no total?", "faturamento_estado", {"uf": "MG"}),
    ("Produtos entre 100 e 500 reais, do mais barato pro mais caro", "preco_entre", {"a": 100, "b": 500}),
    ("Quantos produtos tem na categoria Esporte?", "qtd_produtos_categoria", {"cat": "Esporte"}),
    ("Quantos pedidos por status?", "pedidos_por_status", {}),
]
grupos["manual"] = [{"template": t, "pergunta": p, "sql": SQL_DO[t].format(**pa)} for p, t, pa in MANUAL]

for nome, exemplos in grupos.items():
    with open(DADOS / f"{nome}.jsonl", "w", encoding="utf-8") as f:
        for ex in exemplos:
            ex["prompt"] = prompt(ex["pergunta"])
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"{nome:10s}: {len(exemplos)} exemplos")


def ler(nome):
    with open(DADOS / f"{nome}.jsonl", encoding="utf-8") as f:
        return [json.loads(linha) for linha in f]


ex = ler("treino")[0]
print("\nEXEMPLO DE TREINO:\n" + ex["prompt"] + ex["sql"])
print("\nResultado desse SQL no banco:", executar(ex["sql"])[1][:5])

## ETAPA 6 — Conhecer o modelo base, antes do treinamento
**Por que o Qwen3-0.6B?** Está na lista do professor, tem só 0,6 bilhão de parâmetros (cabe com folga na GPU T4 do Colab, mesmo treinando todos os pesos) e já viu bastante código, inclusive SQL.

In [ ]:
model, tok, device = carregar(BASE)

total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros: {total:,}")

entrada = prompt("Quanto faturamos em março de 2025?")
print("\nPROMPT:\n" + entrada)
print("RESPOSTA DO MODELO BASE:\n" + gerar(model, tok, device, entrada))

## ETAPA 7 — Avaliar o modelo ANTES do treinamento (item 4)
Para cada pergunta do teste: a LLM gera o SQL → o SQL é **executado** no banco → o resultado é comparado com o do SQL correto. Consultas escritas de outro jeito, mas com o mesmo resultado, contam como acerto.

Métricas: **acurácia por execução** (principal), **% de SQL válido** (roda sem erro) e **tempo por pergunta**.

In [ ]:
import pandas as pd

EXPERIMENTOS = []   # tabela com todos os experimentos (base, treinado, parâmetros...)


def avaliar(model, tok, device, exemplos, temperatura=0.0, top_p=1.0, lote=16):
    """Gera o SQL de cada exemplo, executa e compara com o gabarito."""
    model.eval()
    detalhes = []
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inicio = time.time()
    for i in range(0, len(exemplos), lote):
        parte = exemplos[i: i + lote]
        respostas = gerar(model, tok, device, [ex["prompt"] for ex in parte], temperatura, top_p)
        for ex, resp in zip(parte, respostas):
            sql = extrair_sql(resp)
            valido, correto = mesmo_resultado(sql, ex["sql"])
            detalhes.append({"template": ex["template"], "pergunta": ex["pergunta"], "sql_gerado": sql,
                             "sql_correto": ex["sql"], "valido": valido, "correto": correto})
    segundos = time.time() - inicio
    acertos = sum(d["correto"] for d in detalhes)
    return {
        "acuracia": round(100 * acertos / len(detalhes), 1),
        "sql_valido": round(100 * sum(d["valido"] for d in detalhes) / len(detalhes), 1),
        "ms_por_pergunta": round(1000 * segundos / len(detalhes)),
        "acertos": f"{acertos}/{len(detalhes)}",
        "detalhes": detalhes,
    }


def avaliar_e_salvar(model, tok, device, nome, temperatura=0.0, top_p=1.0, conjuntos=("teste", "manual"), limite=None):
    linha = {"experimento": nome, "temperature": temperatura, "top_p": top_p}
    for conjunto in conjuntos:
        exemplos = ler(conjunto)[:limite] if limite else ler(conjunto)
        r = avaliar(model, tok, device, exemplos, temperatura, top_p)
        with open(RESULTADOS / f"{nome}_{conjunto}.json", "w", encoding="utf-8") as f:
            json.dump(r, f, ensure_ascii=False, indent=2)
        linha[f"acuracia_{conjunto}_%"] = r["acuracia"]
        linha[f"acertos_{conjunto}"] = r["acertos"]
        linha[f"sql_valido_{conjunto}_%"] = r["sql_valido"]
        linha["ms_por_pergunta"] = r["ms_por_pergunta"]
        print(f"{nome} | {conjunto:6s}: acurácia {r['acuracia']}% ({r['acertos']}) | "
              f"SQL válido {r['sql_valido']}% | {r['ms_por_pergunta']} ms/pergunta")
    if torch.cuda.is_available():
        linha["vram_pico_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    EXPERIMENTOS.append(linha)
    return linha


resultados_antes = avaliar_e_salvar(model, tok, device, "base")

## ETAPA 8 — Treinar o Qwen3-0.6B na GPU (fine-tuning)
O mesmo laço da aula: **forward → loss → backward → step**, por várias épocas. Só a resposta (o SQL) entra na loss: os tokens do prompt recebem rótulo `-100`.

A cada época, o modelo é avaliado no conjunto de **validação** (acurácia por execução) e o melhor checkpoint é salvo.

In [ ]:
EPOCAS = 3
LOTE = 4
TAXA_APRENDIZADO = 3e-5
MAX_TOKENS = 256


def treinar(model, tok, device, epocas=EPOCAS, lote=LOTE, taxa=TAXA_APRENDIZADO, pasta=TREINADO):
    random.seed(42)
    torch.manual_seed(42)

    dados_treino = ler("treino")       # o teste NÃO participa do treinamento
    dados_validacao = ler("validacao")

    def tokenizar(exemplo):
        ids_prompt = tok(exemplo["prompt"], add_special_tokens=False)["input_ids"]
        ids_resposta = tok(exemplo["sql"], add_special_tokens=False)["input_ids"]
        ids_resposta += [tok.eos_token_id]                       # ensina o modelo a parar
        entrada = ids_prompt + ids_resposta
        rotulos = [-100] * len(ids_prompt) + ids_resposta        # -100: sem loss no prompt
        if len(entrada) > MAX_TOKENS:
            raise ValueError(f"Exemplo com {len(entrada)} tokens. Aumente MAX_TOKENS.")
        return entrada, rotulos

    treino = [tokenizar(ex) for ex in dados_treino]

    def preparar_lote(exemplos):
        tamanho = max(len(e) for e, _ in exemplos)
        entradas, mascaras, rotulos = [], [], []
        for entrada, labels in exemplos:
            faltam = tamanho - len(entrada)
            entradas.append(entrada + [tok.pad_token_id] * faltam)
            mascaras.append([1] * len(entrada) + [0] * faltam)
            rotulos.append(labels + [-100] * faltam)
        return {
            "input_ids": torch.tensor(entradas, device=device),
            "attention_mask": torch.tensor(mascaras, device=device),
            "labels": torch.tensor(rotulos, device=device),
        }

    otimizador = torch.optim.AdamW(model.parameters(), lr=taxa, weight_decay=0.0)
    historico, melhor = [], -1
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    inicio = time.time()

    for epoca in range(1, epocas + 1):
        model.train()
        random.shuffle(treino)
        perdas = []
        for i in range(0, len(treino), lote):
            batch = preparar_lote(treino[i: i + lote])
            loss = model(**batch).loss            # 1. forward + 2. loss
            loss.backward()                        # 3. backward (gradientes)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            otimizador.step()                      # 4. step (ajusta os pesos)
            otimizador.zero_grad()
            perdas.append(loss.item())
            passo = i // lote + 1
            if passo % 50 == 0:
                print(f"  época {epoca} | passo {passo}/{(len(treino) + lote - 1) // lote} | loss {loss.item():.4f}")

        val = avaliar(model, tok, device, dados_validacao)
        loss_media = sum(perdas) / len(perdas)
        historico.append({"epoca": epoca, "loss_treino": round(loss_media, 4),
                          "acuracia_validacao_%": val["acuracia"], "perdas": perdas})
        print(f"ÉPOCA {epoca}: loss {loss_media:.4f} | acurácia na validação {val['acuracia']}% ({val['acertos']})")

        if val["acuracia"] > melhor:   # salva o melhor checkpoint
            melhor = val["acuracia"]
            model.save_pretrained(pasta)
            tok.save_pretrained(pasta)
            print(f"  ✅ checkpoint salvo em {pasta}")

    info = {
        "epocas": epocas, "lote": lote, "taxa": taxa,
        "tempo_min": round((time.time() - inicio) / 60, 1),
        "vram_pico_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None,
        "melhor_acuracia_validacao_%": melhor,
        "historico": historico,
    }
    with open(RESULTADOS / f"treino_{Path(pasta).name}.json", "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"\nTreino concluído em {info['tempo_min']} min | pico de VRAM: {info['vram_pico_gb']} GB")
    return info


info_treino = treinar(model, tok, device)

## ETAPA 9 — Avaliar DEPOIS do treinamento e comparar

In [ ]:
import matplotlib.pyplot as plt

# Libera o modelo atual antes de carregar o melhor checkpoint salvo.
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model, tok, device = carregar(TREINADO)
resultados_depois = avaliar_e_salvar(model, tok, device, "treinado")

print("\nCOMPARAÇÃO ANTES E DEPOIS:")
for nome, r in [("Antes ", resultados_antes), ("Depois", resultados_depois)]:
    print(f"{nome}: teste {r['acuracia_teste_%']}% ({r['acertos_teste']}) | "
          f"manual {r['acuracia_manual_%']}% ({r['acertos_manual']})")

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
rotulos = ["Teste", "Teste manual"]
x = range(len(rotulos))
antes = [resultados_antes["acuracia_teste_%"], resultados_antes["acuracia_manual_%"]]
depois = [resultados_depois["acuracia_teste_%"], resultados_depois["acuracia_manual_%"]]
eixos[0].bar([i - 0.2 for i in x], antes, 0.4, label="Modelo base", color="#e57373")
eixos[0].bar([i + 0.2 for i in x], depois, 0.4, label="Modelo treinado", color="#4caf50")
for i, (a, d) in enumerate(zip(antes, depois)):
    eixos[0].text(i - 0.2, a + 1, f"{a:.0f}%", ha="center")
    eixos[0].text(i + 0.2, d + 1, f"{d:.0f}%", ha="center")
eixos[0].set_xticks(list(x), rotulos)
eixos[0].set_ylim(0, 110)
eixos[0].set_ylabel("Acurácia por execução (%)")
eixos[0].set_title("Antes x depois do fine-tuning")
eixos[0].legend()

perdas = [p for h in info_treino["historico"] for p in h["perdas"]]
eixos[1].plot(perdas, alpha=0.4, label="loss por passo")
janela = 20
media = [sum(perdas[max(0, i - janela):i + 1]) / len(perdas[max(0, i - janela):i + 1]) for i in range(len(perdas))]
eixos[1].plot(media, label="média móvel")
eixos[1].set_xlabel("passo")
eixos[1].set_ylabel("loss")
eixos[1].set_title("Curva de loss do treino")
eixos[1].legend()
plt.tight_layout()
plt.savefig(RESULTADOS / "antes_depois.png", dpi=150)
plt.show()

## ETAPA 10 — Testar diferentes parâmetros (item 3)
### 10.1 Parâmetros de geração: temperature e top_p
`temperature = 0` sempre escolhe o token mais provável. Valores maiores sorteiam tokens menos prováveis, o que deixa a resposta mais "criativa", mas costuma ser ruim para SQL.

In [ ]:
CONFIGS_GERACAO = [
    {"temperatura": 0.0, "top_p": 1.0},
    {"temperatura": 0.3, "top_p": 0.9},
    {"temperatura": 0.7, "top_p": 0.9},
    {"temperatura": 1.0, "top_p": 0.95},
]

torch.manual_seed(0)
for cfg in CONFIGS_GERACAO:
    avaliar_e_salvar(model, tok, device, f"treinado_t{cfg['temperatura']}_p{cfg['top_p']}",
                     cfg["temperatura"], cfg["top_p"], conjuntos=("teste",))

tabela = pd.DataFrame([e for e in EXPERIMENTOS if e["experimento"].startswith("treinado_t")])
tabela[["experimento", "temperature", "top_p", "acuracia_teste_%", "sql_valido_teste_%", "ms_por_pergunta"]]

### 10.2 Parâmetros de treino: taxa de aprendizado e épocas
Treina de novo **a partir do modelo base** com outra configuração e compara com o treino principal. Cada treino leva de 10 a 20 minutos. Se quiser testar mais combinações, mude `TREINOS_EXTRAS`.

In [ ]:
TREINOS_EXTRAS = [
    {"nome": "lr1e-5_1ep", "taxa": 1e-5, "epocas": 1},
    {"nome": "lr1e-4_2ep", "taxa": 1e-4, "epocas": 2},
]

resumo_treinos = [{"experimento": "treinado (principal)", "taxa": TAXA_APRENDIZADO, "epocas": EPOCAS,
                   "tempo_min": info_treino["tempo_min"], "acuracia_teste_%": resultados_depois["acuracia_teste_%"],
                   "acuracia_manual_%": resultados_depois["acuracia_manual_%"]}]

for t in TREINOS_EXTRAS:
    print(f"\n########## Treino extra: {t['nome']} ##########")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model, tok, device = carregar(BASE)
    pasta = PASTA / f"modelo-{t['nome']}"
    info = treinar(model, tok, device, epocas=t["epocas"], taxa=t["taxa"], pasta=pasta)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model, tok, device = carregar(pasta)
    r = avaliar_e_salvar(model, tok, device, t["nome"])
    resumo_treinos.append({"experimento": t["nome"], "taxa": t["taxa"], "epocas": t["epocas"],
                           "tempo_min": info["tempo_min"], "acuracia_teste_%": r["acuracia_teste_%"],
                           "acuracia_manual_%": r["acuracia_manual_%"]})

# Volta para o modelo principal (usado no frontend).
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model, tok, device = carregar(TREINADO)

pd.DataFrame(resumo_treinos)

### 10.3 Tabela com todos os experimentos (cole no README)

In [ ]:
todos = pd.DataFrame(EXPERIMENTOS)
todos.to_csv(RESULTADOS / "experimentos.csv", index=False)
print(todos.to_markdown(index=False))

## ETAPA 11 — Onde o modelo erra? (análise para o relatório)
Compare os erros do modelo base com os do treinado. O base costuma inventar colunas e esquecer a regra dos pedidos cancelados.

In [ ]:
pd.set_option("display.max_colwidth", 180)

for nome in ["base", "treinado"]:
    with open(RESULTADOS / f"{nome}_manual.json", encoding="utf-8") as f:
        detalhes = pd.DataFrame(json.load(f)["detalhes"])
    print(f"\n===== Erros do modelo {nome} no teste manual =====")
    display(detalhes[~detalhes["correto"]][["pergunta", "sql_gerado", "sql_correto"]].head(8))

with open(RESULTADOS / "treinado_teste.json", encoding="utf-8") as f:
    det = pd.DataFrame(json.load(f)["detalhes"])
print("\nAcurácia por tipo de pergunta (modelo treinado):")
display(det.groupby("template")["correto"].mean().mul(100).round(1).sort_values().to_frame("acuracia_%"))

## ETAPA 12 — Fazer uma pergunta e ver o resultado
Mesmo formato do treino: a pergunta vira prompt, a LLM gera o SQL e o Python executa no banco.

In [ ]:
def perguntar(pergunta, temperatura=0.0, top_p=1.0):
    entrada = prompt(pergunta)
    sql = extrair_sql(gerar(model, tok, device, entrada, temperatura, top_p))
    print("PERGUNTA:", pergunta)
    print("SQL GERADO PELA LLM:\n ", sql)
    try:
        colunas, linhas = executar(sql, max_linhas=50)
        print("\nRESULTADO NO BANCO:")
        display(pd.DataFrame(linhas, columns=colunas))
    except Exception as erro:
        print("\nO SQL gerado deu erro:", erro)


# Altere a pergunta para fazer outra consulta.
perguntar("Quais são os 5 produtos mais vendidos?")

## ETAPA 13 — Frontend (item 6)
Rode a célula e clique no link **`https://....gradio.live`** que aparecer. A tela permite fazer perguntas, ajustar temperature e top_p e ver o SQL e a tabela de resultado. **Tire um print da tela** para a entrega. Para parar, interrompa a célula (botão ■).

In [ ]:
import gradio as gr


def responder(pergunta, temperatura, top_p):
    if not pergunta.strip():
        return "Digite uma pergunta.", pd.DataFrame(), ""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inicio = time.time()
    sql = extrair_sql(gerar(model, tok, device, prompt(pergunta), temperatura, top_p))
    tempo = time.time() - inicio
    info = f"⏱️ {tempo:.2f}s · {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
    try:
        colunas, linhas = executar(sql, max_linhas=200)
        return f"```sql\n{sql}\n```", pd.DataFrame(linhas, columns=colunas), f"✅ {len(linhas)} linha(s) · {info}"
    except Exception as erro:
        return f"```sql\n{sql}\n```", pd.DataFrame(), f"❌ Erro ao executar: {erro} · {info}"


with gr.Blocks(title="Assistente SQL") as demo:
    gr.Markdown("# 🧠 Assistente de SQL com LLM\nPergunte em português sobre o banco da loja. "
                "O **Qwen3-0.6B treinado** gera o SQL, que é executado e mostrado abaixo.")
    with gr.Row():
        with gr.Column(scale=3):
            pergunta = gr.Textbox(label="Sua pergunta", placeholder="Ex.: Quanto faturamos em março de 2025?")
            botao = gr.Button("Perguntar", variant="primary")
        with gr.Column(scale=1):
            temperatura = gr.Slider(0.0, 1.5, 0.0, step=0.05, label="temperature")
            top_p = gr.Slider(0.1, 1.0, 1.0, step=0.05, label="top_p")
    sql = gr.Markdown()
    status = gr.Markdown()
    tabela = gr.Dataframe(label="Resultado", interactive=False)
    with gr.Accordion("Tabelas do banco", open=False):
        gr.Code(ESQUEMA)
    gr.Examples(["Quais são os 5 produtos mais vendidos?", "Quanto faturamos em março de 2025?",
                 "Quantos clientes temos em cada estado?", "Quais clientes nunca fizeram um pedido?"],
                inputs=pergunta)
    botao.click(responder, [pergunta, temperatura, top_p], [sql, tabela, status])
    pergunta.submit(responder, [pergunta, temperatura, top_p], [sql, tabela, status])

demo.launch(share=True, debug=True)

## ETAPA 14 — Salvar os resultados
Os arquivos do Colab **somem quando a sessão termina**. Para a entrega:
1. Baixe o zip com os resultados (célula abaixo).
2. Salve este notebook **com as saídas** no GitHub: **Arquivo → Salvar uma cópia no GitHub** → repositório `checkpoint-ia`.

In [ ]:
!zip -qr resultados.zip resultados
from google.colab import files
files.download("resultados.zip")